In [1]:

import pandas as pd
import numpy as np
import os

df = pd.read_csv('electrical_fault_detect_dataset.csv')
print(df.head())
print(df.describe())

   Output (S)          Ia        Ib          Ic        Va        Vb        Vc
0           0 -170.472196  9.219613  161.252583  0.054490 -0.659921  0.605431
1           0 -122.235754  6.168667  116.067087  0.102000 -0.628612  0.526202
2           0  -90.161474  3.813632   86.347841  0.141026 -0.605277  0.464251
3           0  -79.904916  2.398803   77.506112  0.156272 -0.602235  0.445963
4           0  -63.885255  0.590667   63.294587  0.180451 -0.591501  0.411050
         Output (S)            Ia            Ib            Ic            Va  \
count  12001.000000  12001.000000  12001.000000  12001.000000  12001.000000   
mean       0.457962      6.709369    -26.557793     22.353043      0.010517   
std        0.498250    377.158470    357.458613    302.052809      0.346221   
min        0.000000   -883.542316   -900.526951   -883.357762     -0.620748   
25%        0.000000    -64.348986    -51.421937    -54.562257     -0.237610   
50%        0.000000     -3.239788      4.711283     -0.399

In [2]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Daniel Olatunji\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split # Standardization of the data i.e mean = 0 , std = 1 & Data scaling

electrical_cols = ['Ia' , 'Ib' , 'Ic' , 'Va' , 'Vb' , 'Vc']

X = df[electrical_cols].fillna(0) #Fills NaN cells with 0 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

normal_mask = df['Output (S)'] == 0
X_normal = X_scaled[normal_mask]
X_train, X_val = train_test_split(X_normal, test_size=0.2, random_state=42)

In [4]:
%pip install scipy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Daniel Olatunji\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
from scipy.stats import kurtosis, skew

def extract_electrical_features(data):
    # Normalise shape as DataFrame for consistent row-wise operations
    if isinstance(data, np.ndarray):
        data = pd.DataFrame(data, columns=['Ia','Ib','Ic','Va','Vb','Vc'])

    if not isinstance(data, pd.DataFrame):
        raise TypeError('extract_electrical_features expects a pandas DataFrame or numpy array')

    cols = ['Ia', 'Ib', 'Ic', 'Va', 'Vb', 'Vc']
    features = pd.DataFrame(index=data.index)

    # RMS value per channel
    for c in cols:
        features[f'{c}_rms'] = np.sqrt(np.square(data[c]))

    # Current imbalance ratio in the three-phase currents
    features['current_imbalance'] = (
        data[['Ia', 'Ib', 'Ic']].max(axis=1) - data[['Ia', 'Ib', 'Ic']].min(axis=1)
    ) / (data[['Ia', 'Ib', 'Ic']].mean(axis=1) + 1e-6)

    # Statistical features for Ia (scalar over dataset)
    features['kurtosis_Ia'] = kurtosis(data['Ia'].values)
    features['skew_Ia'] = skew(data['Ia'].values)

    return features.values

In [ ]:
%pip install tensorflow
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

input_dim = X_train.shape[1]

input_layer = Input(shape=(input_dim,))
encoded = Dense(64 , activation='relu')(input_layer)
encoded = Dense(32 , activation='relu')(encoded)
encoded = Dense(16 , activation='relu')(encoded) 

decoded = Dense(32 , activation='relu')(encoded)
decoded = Dense(64 , activation='relu')(decoded)
decoded = Dense(input_dim , activation = 'linear')(decoded)

autoencoder = Model(input_layer, decoded)
autoencoder.compile (optimizer= 'adam' , loss = 'mse')

autoencoder.fit(X_train , X_train , 
                epochs=100,
                batch_size = 32,
                validation_data=(X_val , X_val),
                shuffle=True)

autoencoder.save('transformer_electrical_autoencoder.h5')


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Daniel Olatunji\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Epoch 1/100
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.1333 - val_loss: 9.3749e-04
Epoch 2/100
163/163 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 7.9658e-04 - val_loss: 7.3096e-04
Epoch 3/100
163/163 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 7.5065e-04 - val_loss: 7.0271e-04
Epoch 4/100
163/163 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 7.2160e-04 - val_loss: 7.6941e-04
Epoch 5/100
163/163 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 7.0946e-04 - val_loss: 7.4012e-04
Epoch 6/100
163/163 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 7.2218e-04 - val_loss: 6.6310e-04
Epoch 7/100
163/163 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 7.0027e-04 - val_loss: 6.7502e-04
Epoch 8/100
163/163 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 6.7144e-04 - val_loss: 6.6418e-04
Epoch 9/100
163/163 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 6.6500e-04 - val_loss: 6.6998e-04
Epoch

In [ ]:
# Scale the test data
X_test_scaled = scaler.transform(df[electrical_cols].fillna(0))

# Use the autoencoder to reconstruct
reconstructions = autoencoder.predict(X_test_scaled)

# Calculate mean squared error
mse = np.mean(np.power(X_test_scaled - reconstructions, 2), axis=1)

# Compute reconstruction error on validation normal data
recon_val = autoencoder.predict(X_val)
mse_val = np.mean(np.power(X_val - recon_val, 2), axis=1)

# Define threshold based on validation set
threshold = np.mean(mse_val) + 3 * np.std(mse_val)

# Detect anomalies on test data
anomalies = mse > threshold
print(f"Detected {np.sum(anomalies)} potential electrical faults")


376/376 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


NameError: name 'mse_val_normal' is not defined